# Evaluate: GSM8K test set

Colab front-end for `src/eval_gsm8k.py`. Greedy decoding, exact-match on the final numeric answer.

**First run the base model**, then each adapter.

## 1. Mount Drive + Clone + Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RUNS = '/content/drive/MyDrive/finetune-gsm8k-runs'
DRIVE_RESULTS = '/content/drive/MyDrive/finetune-gsm8k-runs/results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if not os.path.isdir('/content/finetune-gsm8k'):
    !git clone https://github.com/YuZh98/finetune-gsm8k.git /content/finetune-gsm8k
os.chdir('/content/finetune-gsm8k')
!pip install -q -r requirements.txt

## 2. Smoke test: 16 problems against the base model

Verify the harness works before running the full 1319 problems.

In [ ]:
from src.eval_gsm8k import evaluate

metrics = evaluate(adapter_path=None, limit=16, batch_size=4)
metrics

## 3. Full eval: base model

In [ ]:
from src.eval_gsm8k import evaluate, append_row
from src.config import ANSWER_REGEX

CSV_PATH = f'{DRIVE_RESULTS}/runs.csv'

metrics = evaluate(adapter_path=None, limit=None, batch_size=8)
append_row(CSV_PATH, {
    'run_id': 'run0_base',
    'adapter': '',
    'answer_regex': ANSWER_REGEX,
    **metrics,
})
print(f'✓ Base: {metrics["accuracy"]:.4f} ({metrics["n_correct"]}/{metrics["n_problems"]})')

## 4. Full eval: trained adapters

Loops over all adapters found on Drive. Or set `RUN_IDS` manually to eval a subset.

In [ ]:
from src.eval_gsm8k import evaluate, append_row
from src.config import ANSWER_REGEX

CSV_PATH = f'{DRIVE_RESULTS}/runs.csv'

# Auto-detect adapters on Drive, or set manually:
# RUN_IDS = ['run1_r8', 'run2_r16', 'run3_r64', 'run4_mlp', 'run5_lr_low', 'run6_lr_high', 'run7_data5k']
RUN_IDS = sorted([
    d for d in os.listdir(DRIVE_RUNS)
    if os.path.isdir(f'{DRIVE_RUNS}/{d}/adapter')
])
print(f'Found adapters: {RUN_IDS}')

for run_id in RUN_IDS:
    adapter = f'{DRIVE_RUNS}/{run_id}/adapter'
    print(f'\nEvaluating {run_id}...')
    metrics = evaluate(adapter_path=adapter, limit=None, batch_size=8)
    append_row(CSV_PATH, {
        'run_id': run_id,
        'adapter': adapter,
        'answer_regex': ANSWER_REGEX,
        **metrics,
    })
    print(f'  ✓ {run_id}: {metrics["accuracy"]:.4f} ({metrics["n_correct"]}/{metrics["n_problems"]})')

## 5. Inspect results

In [ ]:
import pandas as pd

df = pd.read_csv(f'{DRIVE_RESULTS}/runs.csv')
df.sort_values('accuracy', ascending=False)